In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from pathlib import Path

In [3]:
# ==============================
# 1) LOAD AND PREPROCESS GESTURE DATASET
# ==============================4567
csv_path = r"D:\ANN and DL\GestureDataset_110x110x3.csv"  
df = pd.read_csv(csv_path, header=None)  # Assumes first column = label, rest = pixels in [0,1]

# First column is label; rest are pixel values (already normalized to [0,1])
pixel_data = df.iloc[:, 1:].values  # Shape: (N, 36300)

# Validate expected number of pixels
assert pixel_data.shape[1] == 110 * 110 * 3, "Unexpected number of pixel columns!"

# Reshape to (N, 110, 110, 3)
x_data = pixel_data.reshape(-1, 110, 110, 3).astype('float32')

# Optional safety check: ensure values are in [0, 1]
assert x_data.min() >= 0.0 and x_data.max() <= 1.0, "Pixel values must be in [0,1] for sigmoid output!"

# Train/test split (90% train, 10% test — test unused but defined for clarity)
num_samples = x_data.shape[0]
split_idx = int(0.9 * num_samples)
x_train = x_data[:split_idx]
# x_test = x_data[split_idx:]  # unused in GAN

print(f"Loaded {x_train.shape[0]} training images of shape {x_train.shape[1:]}")

Loaded 2700 training images of shape (110, 110, 3)


In [4]:
# ==============================
# 2) HYPERPARAMETERS
# ==============================
noise_dim = 100
num_examples_to_generate = 16
batch_size = 16  # Reduced for memory (110x110x3 is large!)
epochs = 10

In [5]:
# ==============================
# 3) BUILD GENERATOR (outputs 110x110x3)
# ==============================
def build_generator():
    model = models.Sequential([
        layers.Dense(7 * 7 * 512, input_shape=(noise_dim,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((7, 7, 512)),

        # 7x7 → 14x14
        layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # 14x14 → 28x28
        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # 28x28 → 56x56
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # 56x56 → 112x112
        layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', activation='sigmoid'),

        # Crop from 112x112 to 110x110
        layers.Lambda(lambda x: x[:, 1:-1, 1:-1, :])
    ], name="Generator")
    return model

In [6]:
# ==============================
# 4) BUILD DISCRIMINATOR (inputs 110x110x3)
# ==============================
def build_discriminator():
    model = models.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[110, 110, 3]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ], name="Discriminator")
    return model

In [7]:
# ==============================
# 5) INSTANTIATE MODELS & OPTIMIZERS
# ==============================
generator = build_generator()
discriminator = build_discriminator()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# Print model summaries (optional but helpful)
# generator.summary()
# discriminator.summary()

In [8]:
# ==============================
# 6) LOSS FUNCTIONS
# ==============================
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

In [9]:
# ==============================
# 7) IMAGE GENERATION HELPER
# ==============================
def generate_and_save_images(model, epoch):
    noise = tf.random.normal([num_examples_to_generate, noise_dim])
    generated_images = model(noise, training=False)

    plt.figure(figsize=(6, 6))
    for i in range(generated_images.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(generated_images[i])  # Already in [0,1], RGB
        plt.axis('off')
    save_dir = Path("generated_images")
    save_dir.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_dir / f"image_at_epoch_{epoch:04d}.png", dpi=150)
    plt.close()

In [10]:
# ==============================
# 8) PREPARE TF DATASET
# ==============================
train_dataset = tf.data.Dataset.from_tensor_slices(x_train)
train_dataset = train_dataset.shuffle(buffer_size=10000).batch(batch_size)

In [11]:
# ==============================
# 9) TRAINING LOOP
# ==============================
def train(dataset, epochs):
    for epoch in range(epochs):
        gen_loss_avg = tf.keras.metrics.Mean()
        disc_loss_avg = tf.keras.metrics.Mean()

        for image_batch in dataset:
            curr_batch_size = tf.shape(image_batch)[0]
            noise = tf.random.normal([curr_batch_size, noise_dim])

            # Train Discriminator
            with tf.GradientTape() as disc_tape:
                generated_images = generator(noise, training=True)
                real_output = discriminator(image_batch, training=True)
                fake_output = discriminator(generated_images, training=True)
                disc_loss = discriminator_loss(real_output, fake_output)

            grads_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
            discriminator_optimizer.apply_gradients(zip(grads_disc, discriminator.trainable_variables))

            # Train Generator
            noise = tf.random.normal([curr_batch_size, noise_dim])
            with tf.GradientTape() as gen_tape:
                generated_images = generator(noise, training=True)
                fake_output = discriminator(generated_images, training=True)
                gen_loss = generator_loss(fake_output)

            grads_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
            generator_optimizer.apply_gradients(zip(grads_gen, generator.trainable_variables))

            gen_loss_avg.update_state(gen_loss)
            disc_loss_avg.update_state(disc_loss)

        print(f'Epoch {epoch + 1}, '
              f'Generator Loss: {gen_loss_avg.result():.4f}, '
              f'Discriminator Loss: {disc_loss_avg.result():.4f}')
        generate_and_save_images(generator, epoch + 1)

In [12]:
# ==============================
# 10) RUN TRAINING
# ==============================
if __name__ == "__main__":
    train(train_dataset, epochs)

Epoch 1, Generator Loss: 2.1462, Discriminator Loss: 1.0685
Epoch 2, Generator Loss: 3.4486, Discriminator Loss: 1.2126
Epoch 3, Generator Loss: 5.7921, Discriminator Loss: 1.0178
Epoch 4, Generator Loss: 2.1929, Discriminator Loss: 0.6469
Epoch 5, Generator Loss: 1.8448, Discriminator Loss: 0.9442
Epoch 6, Generator Loss: 1.7738, Discriminator Loss: 1.0869
Epoch 7, Generator Loss: 1.6440, Discriminator Loss: 0.9858
Epoch 8, Generator Loss: 1.9418, Discriminator Loss: 0.8792
Epoch 9, Generator Loss: 1.3846, Discriminator Loss: 1.1533
Epoch 10, Generator Loss: 1.3839, Discriminator Loss: 1.0789
